In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 27 — LangGraph Supplier Risk Agent
# Purpose:
# Build an agentic supplier-risk assistant that:
# 1. Resolves supplier from a user question
# 2. Reads gold_chat_context_snapshots first
# 3. Retrieves supplier-specific RAG evidence
# 4. Reranks + cleans evidence
# 5. Generates a grounded supplier-risk answer
# 6. Saves agent run logs
# ─────────────────────────────────────────────────────────────

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    IntegerType,
    TimestampType
)

import json
import re
import numpy as np
import pandas as pd
from datetime import datetime

# ─────────────────────────────────────────────────────────────
# Gold table config
# ─────────────────────────────────────────────────────────────

GOLD_SCHEMA = "supplysage_gold"

CHAT_CONTEXT_TABLE = f"{GOLD_SCHEMA}.gold_chat_context_snapshots"
EMBEDDINGS_TABLE = f"{GOLD_SCHEMA}.gold_rag_embeddings"
RETRIEVAL_INDEX_TABLE = f"{GOLD_SCHEMA}.gold_rag_retrieval_index"
RETRIEVAL_TEST_TABLE = f"{GOLD_SCHEMA}.gold_rag_retrieval_test_results"

AGENT_RUN_LOG_TABLE = f"{GOLD_SCHEMA}.gold_supplier_risk_agent_run_logs"

# Same embedding model used in Notebook 25 and Notebook 26
ST_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_DIM = 384

# Retrieval settings
TOP_K_CANDIDATES = 2000
TOP_K_FINAL = 8

print("Notebook 27 config loaded.")
print(f"CHAT_CONTEXT_TABLE: {CHAT_CONTEXT_TABLE}")
print(f"EMBEDDINGS_TABLE: {EMBEDDINGS_TABLE}")
print(f"RETRIEVAL_INDEX_TABLE: {RETRIEVAL_INDEX_TABLE}")
print(f"AGENT_RUN_LOG_TABLE: {AGENT_RUN_LOG_TABLE}")

# ─────────────────────────────────────────────────────────────
# Validate required tables
# ─────────────────────────────────────────────────────────────

required_tables = [
    CHAT_CONTEXT_TABLE,
    EMBEDDINGS_TABLE,
    RETRIEVAL_INDEX_TABLE,
]

for table_name in required_tables:
    print(f"\nChecking table: {table_name}")
    df = spark.table(table_name)
    row_count = df.count()
    print(f"Rows: {row_count}")
    assert row_count > 0, f"{table_name} is empty."

# Strong expected counts from previous notebooks
chat_count = spark.table(CHAT_CONTEXT_TABLE).count()
embedding_count = spark.table(EMBEDDINGS_TABLE).count()
index_count = spark.table(RETRIEVAL_INDEX_TABLE).count()

assert chat_count == 48, f"Expected 48 supplier chat snapshots, got {chat_count}"
assert embedding_count == index_count, "Embedding table and retrieval index row counts do not match."

print("\nTable checks passed.")
print(f"Supplier snapshots: {chat_count}")
print(f"Embedding rows: {embedding_count}")
print(f"Retrieval index rows: {index_count}")

# Preview high-risk / highest-score suppliers available for testing
display(
    spark.table(CHAT_CONTEXT_TABLE)
    .select(
        "supplier_id",
        "supplier_name",
        "supplier_risk_score",
        "risk_band",
        "active_external_event_count",
        "top_affected_sku_count",
        "final_top_risk_driver",
        "final_recommended_action"
    )
    .orderBy(F.desc("supplier_risk_score"))
    .limit(10)
)

In [0]:
# ─────────────────────────────────────────────────────────────
# Cell 2 — Supplier resolver
# Resolves supplier_id from user question using:
# 1. Explicit SUP_### pattern
# 2. Supplier name substring match
# 3. Highest text similarity fallback using simple token overlap
# ─────────────────────────────────────────────────────────────

from pyspark.sql import functions as F
import re
import json

CHAT_CONTEXT_TABLE = "supplysage_gold.gold_chat_context_snapshots"

supplier_lookup_pd = (
    spark.table(CHAT_CONTEXT_TABLE)
    .select(
        "supplier_id",
        "supplier_name",
        "supplier_risk_score",
        "risk_band",
        "final_top_risk_driver",
        "final_recommended_action"
    )
    .toPandas()
)

supplier_lookup_pd["supplier_id_lower"] = supplier_lookup_pd["supplier_id"].astype(str).str.lower()
supplier_lookup_pd["supplier_name_lower"] = supplier_lookup_pd["supplier_name"].fillna("").astype(str).str.lower()

print(f"Loaded {len(supplier_lookup_pd)} suppliers into resolver.")

def normalize_text(text):
    if text is None:
        return ""
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9_\s-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def token_overlap_score(query, candidate):
    query_tokens = set(normalize_text(query).split())
    candidate_tokens = set(normalize_text(candidate).split())

    if not query_tokens or not candidate_tokens:
        return 0.0

    overlap = len(query_tokens.intersection(candidate_tokens))
    return overlap / max(len(candidate_tokens), 1)

def resolve_supplier_from_question(question):
    """
    Resolve supplier from natural language question.
    Returns a dict with supplier_id, supplier_name, confidence, and resolution_method.
    """

    if question is None or len(str(question).strip()) == 0:
        return {
            "supplier_id": None,
            "supplier_name": None,
            "confidence": 0.0,
            "resolution_method": "empty_question",
            "message": "No question provided."
        }

    q = str(question).strip()
    q_norm = normalize_text(q)

    # 1. Explicit supplier ID pattern
    id_match = re.search(r"\bSUP[_-]?\d+\b", q, flags=re.IGNORECASE)
    if id_match:
        raw_id = id_match.group(0).upper().replace("-", "_")
        if not raw_id.startswith("SUP_"):
            raw_id = raw_id.replace("SUP", "SUP_")

        match_df = supplier_lookup_pd[
            supplier_lookup_pd["supplier_id"].astype(str).str.upper() == raw_id
        ]

        if len(match_df) > 0:
            row = match_df.iloc[0]
            return {
                "supplier_id": row["supplier_id"],
                "supplier_name": row["supplier_name"],
                "confidence": 1.0,
                "resolution_method": "explicit_supplier_id",
                "message": f"Resolved explicit supplier ID {raw_id}."
            }

    # 2. Direct supplier name substring match
    name_matches = []

    for _, row in supplier_lookup_pd.iterrows():
        supplier_name = str(row["supplier_name"] or "")
        supplier_name_norm = normalize_text(supplier_name)

        if supplier_name_norm and supplier_name_norm in q_norm:
            name_matches.append((row, 0.95, "supplier_name_exact_substring"))

        # Also match first significant token for names like FlexPack
        first_token = supplier_name_norm.split()[0] if supplier_name_norm else ""
        if first_token and len(first_token) >= 5 and first_token in q_norm:
            name_matches.append((row, 0.85, "supplier_name_first_token"))

    if name_matches:
        row, conf, method = sorted(name_matches, key=lambda x: x[1], reverse=True)[0]
        return {
            "supplier_id": row["supplier_id"],
            "supplier_name": row["supplier_name"],
            "confidence": conf,
            "resolution_method": method,
            "message": f"Resolved supplier by name: {row['supplier_name']}."
        }

    # 3. Token overlap fallback
    scored = []

    for _, row in supplier_lookup_pd.iterrows():
        candidate_text = f"{row['supplier_id']} {row['supplier_name']}"
        score = token_overlap_score(q, candidate_text)
        scored.append((row, score))

    best_row, best_score = sorted(scored, key=lambda x: x[1], reverse=True)[0]

    if best_score > 0:
        return {
            "supplier_id": best_row["supplier_id"],
            "supplier_name": best_row["supplier_name"],
            "confidence": float(best_score),
            "resolution_method": "token_overlap",
            "message": f"Resolved supplier by token overlap: {best_row['supplier_name']}."
        }

    return {
        "supplier_id": None,
        "supplier_name": None,
        "confidence": 0.0,
        "resolution_method": "unresolved",
        "message": "Could not resolve supplier from question."
    }

# Test resolver
test_questions = [
    "Why is supplier SUP_134 risky?",
    "What is happening with FlexPack Industries?",
    "Explain risk for FlexPack",
    "Which supplier should I monitor?"
]

for q in test_questions:
    print("\nQuestion:", q)
    print(resolve_supplier_from_question(q))

In [0]:
# ─────────────────────────────────────────────────────────────
# Cell 3 — Tool 1: get_supplier_snapshot
# Reads gold_chat_context_snapshots first.
# This is the agent's primary supplier context lookup.
# ─────────────────────────────────────────────────────────────

import json
from pyspark.sql import functions as F

CHAT_CONTEXT_TABLE = "supplysage_gold.gold_chat_context_snapshots"

def get_supplier_snapshot(supplier_id: str) -> dict:
    """
    Fetch one supplier-level chatbot context snapshot.
    Returns structured supplier facts plus preformatted chat_context_text.
    """

    if supplier_id is None or len(str(supplier_id).strip()) == 0:
        return {
            "ok": False,
            "error": "supplier_id is required.",
            "supplier_id": None,
            "snapshot": None
        }

    supplier_id = str(supplier_id).strip().upper()

    rows = (
        spark.table(CHAT_CONTEXT_TABLE)
        .filter(F.col("supplier_id") == supplier_id)
        .select(
            "supplier_id",
            "supplier_name",
            "snapshot_date",
            "snapshot_timestamp",
            "supplier_risk_score",
            "risk_band",
            "score_delta",
            "criticality_tier",
            "annual_spend",
            "mapped_sku_count",
            "open_alert_count",
            "critical_or_high_alert_count",
            "active_external_event_count",
            "high_severity_event_count",
            "top_affected_sku_count",
            "high_risk_sku_count",
            "latest_external_event_date",
            "final_top_risk_driver",
            "final_recommended_action",
            "chat_context_json",
            "chat_context_text"
        )
        .limit(1)
        .collect()
    )

    if len(rows) == 0:
        return {
            "ok": False,
            "error": f"No supplier snapshot found for {supplier_id}.",
            "supplier_id": supplier_id,
            "snapshot": None
        }

    row = rows[0]

    try:
        snapshot_json = json.loads(row["chat_context_json"])
    except Exception as e:
        snapshot_json = {}
        json_error = str(e)
    else:
        json_error = None

    snapshot = {
        "supplier_id": row["supplier_id"],
        "supplier_name": row["supplier_name"],
        "snapshot_date": str(row["snapshot_date"]),
        "snapshot_timestamp": str(row["snapshot_timestamp"]),
        "supplier_risk_score": float(row["supplier_risk_score"]) if row["supplier_risk_score"] is not None else None,
        "risk_band": row["risk_band"],
        "score_delta": float(row["score_delta"]) if row["score_delta"] is not None else None,
        "criticality_tier": row["criticality_tier"],
        "annual_spend": float(row["annual_spend"]) if row["annual_spend"] is not None else None,
        "mapped_sku_count": int(row["mapped_sku_count"]) if row["mapped_sku_count"] is not None else 0,
        "open_alert_count": int(row["open_alert_count"]) if row["open_alert_count"] is not None else 0,
        "critical_or_high_alert_count": int(row["critical_or_high_alert_count"]) if row["critical_or_high_alert_count"] is not None else 0,
        "active_external_event_count": int(row["active_external_event_count"]) if row["active_external_event_count"] is not None else 0,
        "high_severity_event_count": int(row["high_severity_event_count"]) if row["high_severity_event_count"] is not None else 0,
        "top_affected_sku_count": int(row["top_affected_sku_count"]) if row["top_affected_sku_count"] is not None else 0,
        "high_risk_sku_count": int(row["high_risk_sku_count"]) if row["high_risk_sku_count"] is not None else 0,
        "latest_external_event_date": str(row["latest_external_event_date"]) if row["latest_external_event_date"] is not None else None,
        "top_risk_driver": row["final_top_risk_driver"],
        "recommended_action": row["final_recommended_action"],
        "chat_context_text": row["chat_context_text"],
        "chat_context_json": snapshot_json,
        "chat_context_json_error": json_error
    }

    return {
        "ok": True,
        "error": None,
        "supplier_id": supplier_id,
        "snapshot": snapshot
    }


# Test Tool 1
snapshot_result = get_supplier_snapshot("SUP_134")

print("Snapshot lookup result:")
print("ok:", snapshot_result["ok"])
print("error:", snapshot_result["error"])

if snapshot_result["ok"]:
    s = snapshot_result["snapshot"]
    print("supplier_id:", s["supplier_id"])
    print("supplier_name:", s["supplier_name"])
    print("risk_score:", s["supplier_risk_score"])
    print("risk_band:", s["risk_band"])
    print("top_risk_driver:", s["top_risk_driver"])
    print("recommended_action:", s["recommended_action"])
    print("active_external_event_count:", s["active_external_event_count"])
    print("top_affected_sku_count:", s["top_affected_sku_count"])
    print("\nChat context preview:")
    print(s["chat_context_text"][:1500])

In [0]:
# ─────────────────────────────────────────────────────────────
# Cell 4 — Load embedding model for agent retrieval
# Same model as Notebook 25 and 26
# ─────────────────────────────────────────────────────────────

import subprocess
import numpy as np

# Install only if needed
subprocess.run(
    ["pip", "install", "sentence-transformers", "--quiet"],
    check=True
)

from sentence_transformers import SentenceTransformer

ST_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

print(f"Loading retrieval embedding model: {ST_MODEL_NAME}")

retrieval_model = SentenceTransformer(ST_MODEL_NAME)

EMBEDDING_DIM = int(
    retrieval_model.get_embedding_dimension()
    if hasattr(retrieval_model, "get_embedding_dimension")
    else retrieval_model.get_sentence_embedding_dimension()
)

print(f"Model loaded. Embedding dimension: {EMBEDDING_DIM}")

assert EMBEDDING_DIM == 384, f"Expected 384 dimensions, got {EMBEDDING_DIM}"

print("Embedding model ready for agent retrieval.")

In [0]:
# ─────────────────────────────────────────────────────────────
# Cell 5 — Tool 2: retrieve_supplier_evidence
# Retrieves supplier-specific evidence from gold_rag_embeddings
# using query embedding + business-aware reranking.
# ─────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
from pyspark.sql import functions as F

EMBEDDINGS_TABLE = "supplysage_gold.gold_rag_embeddings"

TOP_K_CANDIDATES = 2000
TOP_K_FINAL = 8

def retrieve_supplier_evidence(
    supplier_id: str,
    question: str,
    top_k_candidates: int = TOP_K_CANDIDATES,
    top_k_final: int = TOP_K_FINAL
) -> dict:
    """
    Retrieve and rerank supplier-specific evidence.
    Returns cleaned, ranked evidence records for the agent.
    """

    if supplier_id is None or len(str(supplier_id).strip()) == 0:
        return {
            "ok": False,
            "error": "supplier_id is required.",
            "supplier_id": supplier_id,
            "evidence": []
        }

    if question is None or len(str(question).strip()) == 0:
        return {
            "ok": False,
            "error": "question is required.",
            "supplier_id": supplier_id,
            "evidence": []
        }

    supplier_id = str(supplier_id).strip().upper()
    question = str(question).strip()

    # Embed query
    query_embedding = retrieval_model.encode(
        question,
        normalize_embeddings=True
    ).tolist()

    query_vector = np.array(query_embedding, dtype=np.float32)

    # Pull supplier-specific candidate evidence.
    # Order first by freshness/event date to avoid pulling old noisy chunks first.
    candidate_pd = (
        spark.table(EMBEDDINGS_TABLE)
        .filter(F.col("supplier_id") == supplier_id)
        .select(
            "chunk_id",
            "original_chunk_id",
            "chunk_type",
            "supplier_id",
            "sku_id",
            "source_name",
            "risk_category",
            "event_date",
            "severity",
            "freshness_weight",
            "source_url",
            "evidence_doc_id",
            "chunk_text",
            "embedding"
        )
        .orderBy(
            F.desc("freshness_weight"),
            F.desc("event_date")
        )
        .limit(top_k_candidates)
        .toPandas()
    )

    if len(candidate_pd) == 0:
        return {
            "ok": False,
            "error": f"No evidence found for supplier {supplier_id}.",
            "supplier_id": supplier_id,
            "candidate_count": 0,
            "evidence": []
        }

    # Convert embeddings to matrix
    embedding_matrix = np.vstack(
        candidate_pd["embedding"]
        .apply(lambda x: np.array(x, dtype=np.float32))
        .values
    )

    if embedding_matrix.shape[1] != len(query_vector):
        return {
            "ok": False,
            "error": f"Embedding dimension mismatch. Evidence dim={embedding_matrix.shape[1]}, query dim={len(query_vector)}",
            "supplier_id": supplier_id,
            "candidate_count": len(candidate_pd),
            "evidence": []
        }

    # Cosine similarity because both query/evidence embeddings are normalized
    candidate_pd["cosine_similarity"] = embedding_matrix @ query_vector

    # Freshness
    candidate_pd["freshness_weight"] = pd.to_numeric(
        candidate_pd["freshness_weight"],
        errors="coerce"
    ).fillna(0.5)

    candidate_pd["event_date_parsed"] = pd.to_datetime(
        candidate_pd["event_date"],
        errors="coerce"
    )

    today = pd.Timestamp.today().normalize()

    candidate_pd["days_since_event"] = (
        today - candidate_pd["event_date_parsed"]
    ).dt.days

    candidate_pd["recency_score"] = np.select(
        [
            candidate_pd["event_date_parsed"].isna(),
            candidate_pd["days_since_event"] <= 30,
            candidate_pd["days_since_event"] <= 90,
            candidate_pd["days_since_event"] <= 365,
        ],
        [
            0.35,
            1.00,
            0.85,
            0.65,
        ],
        default=0.25
    )

    # Severity
    severity_map = {
        "critical": 1.00,
        "high": 0.90,
        "medium": 0.55,
        "low": 0.30
    }

    candidate_pd["severity_score"] = (
        candidate_pd["severity"]
        .fillna("unknown")
        .astype(str)
        .str.lower()
        .map(severity_map)
        .fillna(0.40)
    )

    # Chunk type score
    candidate_pd["chunk_type_score"] = np.select(
        [
            candidate_pd["chunk_type"].astype(str).str.lower().eq("risk_explanation"),
            candidate_pd["chunk_type"].astype(str).str.lower().eq("external_event"),
        ],
        [
            1.00,
            0.85,
        ],
        default=0.50
    )

    # Prefer business-critical sources
    driver_sources = ["internal_risk_engine", "ofac", "cisa", "nws", "eia", "sec", "cpsc"]

    candidate_pd["driver_source_score"] = (
        candidate_pd["source_name"]
        .fillna("")
        .astype(str)
        .str.lower()
        .isin(driver_sources)
        .astype(float)
    )

    # Prefer useful categories
    driver_categories = [
        "supplier_risk",
        "sanctions_compliance",
        "cyber",
        "logistics",
        "operational",
        "quality_recall",
        "weather",
        "energy"
    ]

    candidate_pd["driver_category_score"] = (
        candidate_pd["risk_category"]
        .fillna("")
        .astype(str)
        .str.lower()
        .isin(driver_categories)
        .astype(float)
    )

    # Final business-aware score
    candidate_pd["business_retrieval_score"] = (
        0.45 * candidate_pd["cosine_similarity"]
        + 0.15 * candidate_pd["freshness_weight"]
        + 0.15 * candidate_pd["severity_score"]
        + 0.10 * candidate_pd["recency_score"]
        + 0.10 * candidate_pd["driver_source_score"]
        + 0.05 * candidate_pd["driver_category_score"]
    )

    ranked_pd = (
        candidate_pd
        .sort_values(
            by=[
                "business_retrieval_score",
                "driver_source_score",
                "severity_score",
                "recency_score",
                "cosine_similarity"
            ],
            ascending=False
        )
        .head(top_k_final)
        .copy()
        .reset_index(drop=True)
    )

    evidence = []

    for idx, row in ranked_pd.iterrows():
        evidence.append({
            "rank": int(idx + 1),
            "chunk_id": row.get("chunk_id"),
            "original_chunk_id": row.get("original_chunk_id"),
            "chunk_type": row.get("chunk_type"),
            "supplier_id": row.get("supplier_id"),
            "sku_id": row.get("sku_id"),
            "source_name": row.get("source_name"),
            "risk_category": row.get("risk_category"),
            "event_date": str(row.get("event_date")) if pd.notna(row.get("event_date")) else None,
            "severity": row.get("severity"),
            "freshness_weight": float(row.get("freshness_weight")) if pd.notna(row.get("freshness_weight")) else None,
            "cosine_similarity": float(row.get("cosine_similarity")) if pd.notna(row.get("cosine_similarity")) else None,
            "business_retrieval_score": float(row.get("business_retrieval_score")) if pd.notna(row.get("business_retrieval_score")) else None,
            "source_url": row.get("source_url"),
            "evidence_doc_id": row.get("evidence_doc_id"),
            "chunk_text": row.get("chunk_text")
        })

    return {
        "ok": True,
        "error": None,
        "supplier_id": supplier_id,
        "question": question,
        "candidate_count": int(len(candidate_pd)),
        "returned_count": int(len(evidence)),
        "evidence": evidence
    }


# Test Tool 2
test_question = """
Why is supplier SUP_134 high risk?
Explain the main external events, affected SKUs, and recommended action.
"""

evidence_result = retrieve_supplier_evidence(
    supplier_id="SUP_134",
    question=test_question,
    top_k_candidates=2000,
    top_k_final=8
)

print("Evidence retrieval result:")
print("ok:", evidence_result["ok"])
print("error:", evidence_result["error"])
print("supplier_id:", evidence_result["supplier_id"])
print("candidate_count:", evidence_result.get("candidate_count"))
print("returned_count:", evidence_result.get("returned_count"))

for ev in evidence_result["evidence"][:5]:
    print("\n" + "-" * 100)
    print(
        f"Rank {ev['rank']} | "
        f"Score {ev['business_retrieval_score']:.4f} | "
        f"{ev['source_name']} | {ev['risk_category']} | "
        f"{ev['severity']} | {ev['event_date']}"
    )
    print(str(ev["chunk_text"])[:700])

In [0]:
# ─────────────────────────────────────────────────────────────
# Cell 6 — Tool 3: clean_retrieved_evidence
# Cleans noisy raw payloads before the agent uses evidence.
# Especially important for OFAC XML / JSON payloads.
# ─────────────────────────────────────────────────────────────

import re
import pandas as pd

def clean_evidence_text(text, max_chars: int = 700) -> str:
    """
    Clean evidence text for chatbot/agent context.
    Keeps readable risk text and removes large raw XML/JSON payloads.
    """

    if text is None:
        return ""

    text = str(text)

    # Detect raw JSON payloads with raw_text
    if '"raw_text"' in text or "'raw_text'" in text:
        raw_text_pos = text.find('"raw_text"')
        if raw_text_pos == -1:
            raw_text_pos = text.find("'raw_text'")

        prefix = text[:raw_text_pos].strip()

        # Keep readable source/title prefix if available
        if len(prefix) > 30:
            text = prefix + " [Raw payload omitted for readability.]"
        else:
            text = "Raw sanctions/compliance payload detected. Full payload omitted for readability."

    # Remove XML blocks if they slipped through
    text = re.sub(
        r"<\?xml.*",
        "[Raw XML payload omitted for readability.]",
        text,
        flags=re.DOTALL
    )

    # Remove remaining XML/HTML-like tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Normalize whitespace
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    # Truncate
    if len(text) > max_chars:
        text = text[:max_chars].rstrip() + "..."

    return text


def clean_retrieved_evidence(evidence_result: dict, max_chars: int = 700) -> dict:
    """
    Takes output from retrieve_supplier_evidence and returns cleaned evidence records.
    """

    if not evidence_result.get("ok"):
        return {
            "ok": False,
            "error": evidence_result.get("error", "Evidence retrieval failed."),
            "supplier_id": evidence_result.get("supplier_id"),
            "question": evidence_result.get("question"),
            "evidence": []
        }

    cleaned_records = []

    for ev in evidence_result.get("evidence", []):
        cleaned_records.append({
            "rank": ev.get("rank"),
            "chunk_id": ev.get("chunk_id"),
            "original_chunk_id": ev.get("original_chunk_id"),
            "chunk_type": ev.get("chunk_type"),
            "supplier_id": ev.get("supplier_id"),
            "sku_id": ev.get("sku_id"),
            "source_name": ev.get("source_name"),
            "risk_category": ev.get("risk_category"),
            "event_date": ev.get("event_date"),
            "severity": ev.get("severity"),
            "freshness_weight": ev.get("freshness_weight"),
            "cosine_similarity": ev.get("cosine_similarity"),
            "business_retrieval_score": ev.get("business_retrieval_score"),
            "source_url": ev.get("source_url"),
            "evidence_doc_id": ev.get("evidence_doc_id"),
            "cleaned_evidence_text": clean_evidence_text(
                ev.get("chunk_text"),
                max_chars=max_chars
            )
        })

    return {
        "ok": True,
        "error": None,
        "supplier_id": evidence_result.get("supplier_id"),
        "question": evidence_result.get("question"),
        "candidate_count": evidence_result.get("candidate_count"),
        "returned_count": len(cleaned_records),
        "evidence": cleaned_records
    }


# Test Tool 3
cleaned_evidence_result = clean_retrieved_evidence(evidence_result)

print("Cleaned evidence result:")
print("ok:", cleaned_evidence_result["ok"])
print("error:", cleaned_evidence_result["error"])
print("supplier_id:", cleaned_evidence_result["supplier_id"])
print("returned_count:", cleaned_evidence_result["returned_count"])

for ev in cleaned_evidence_result["evidence"][:6]:
    print("\n" + "-" * 100)
    print(
        f"Rank {ev['rank']} | "
        f"Score {ev['business_retrieval_score']:.4f} | "
        f"{ev['source_name']} | {ev['risk_category']} | "
        f"{ev['severity']} | {ev['event_date']}"
    )
    print(ev["cleaned_evidence_text"])

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 27 — Cell 7 (UPGRADED): LLM-powered RAG answer generator
#
# This replaces the deterministic generate_supplier_risk_answer with an LLM
# node that REASONS over your retrieved evidence. The retrieval ("R") is
# unchanged — this is the "augment + generate" ("AG") that completes RAG.
#
# Grounding: the LLM is given ONLY the supplier snapshot + retrieved evidence,
# and instructed to answer strictly from that context. This keeps answers
# grounded in your pipeline's data and reduces hallucination.
#
# Model: Databricks Foundation Model API, Llama 3.3 70B (pay-per-token).
#   Endpoint: databricks-meta-llama-3-3-70b-instruct
#   If that endpoint is unavailable on your account/region, set
#   LLM_ENDPOINT to one you CAN query (check the AI Playground), or set
#   USE_LLM = False to fall back to the deterministic generator below.
# ─────────────────────────────────────────────────────────────

import json

USE_LLM = True
# Confirmed Ready in your workspace (AI Gateway). Swap to another listed endpoint
# any time, e.g. "databricks-claude-opus-4-8" or "databricks-gemini-3-5-flash".
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"

# Free Edition token budgeting:
#   Every call draws DBUs (Llama 3.3 70B: ~7.1 in / ~21.4 out per 1M tokens) against
#   your daily quota. Keep output modest and trim context to control spend.
LLM_MAX_TOKENS = 600          # cap answer length (output tokens are the pricey ones)
LLM_TEMPERATURE = 0.2         # low = factual, grounded, repeatable
MAX_EVIDENCE_ITEMS = 5        # how many retrieved chunks to inject (fewer = cheaper)
MAX_EVIDENCE_CHARS = 500      # truncate each chunk (raw payloads can be huge)


def _get_llm_client():
    """Databricks Foundation Model client via the OpenAI-compatible SDK.
    Uses the workspace's own token/host when run inside Databricks."""
    from databricks.sdk import WorkspaceClient
    w = WorkspaceClient()
    # openai-compatible client pointed at the serving endpoints route
    return w.serving_endpoints.get_open_ai_client()


def _build_grounding_context(snapshot: dict, snapshot_json: dict, evidence: list) -> str:
    """Assemble the retrieved evidence + snapshot into a compact context block."""
    lines = []
    lines.append("SUPPLIER SNAPSHOT (authoritative):")
    lines.append(f"- supplier_id: {snapshot.get('supplier_id')}")
    lines.append(f"- supplier_name: {snapshot.get('supplier_name')}")
    lines.append(f"- risk_score: {snapshot.get('supplier_risk_score')}")
    lines.append(f"- risk_band: {snapshot.get('risk_band')}")
    lines.append(f"- top_risk_driver: {snapshot.get('top_risk_driver')}")
    lines.append(f"- recommended_action: {snapshot.get('recommended_action')}")

    events = snapshot_json.get("active_external_events") or []
    if events:
        lines.append("\nACTIVE EXTERNAL EVENTS:")
        for e in events[:8]:
            lines.append(
                f"- {e.get('source_name')} | {e.get('risk_category')} | "
                f"{e.get('severity')} | {e.get('event_date')}: {e.get('event_title')}"
            )

    skus = snapshot_json.get("top_affected_skus") or []
    if skus:
        lines.append("\nTOP AFFECTED SKUS:")
        for s in skus[:6]:
            lines.append(
                f"- {s.get('canonical_sku_id')}: dependency={s.get('dependency_percent')}, "
                f"stockout_risk={s.get('stockout_risk_score')}, "
                f"primary={s.get('is_primary_supplier')}, alt={s.get('alternate_supplier_id')}"
            )

    if evidence:
        lines.append("\nRETRIEVED EVIDENCE (RAG):")
        for i, ev in enumerate(evidence[:MAX_EVIDENCE_ITEMS], start=1):
            txt = (ev.get("cleaned_evidence_text") or "").strip()
            lines.append(f"[{i}] {txt[:MAX_EVIDENCE_CHARS]}")

    return "\n".join(lines)


SYSTEM_PROMPT = (
    "You are SupplySage AI, a supplier-risk analyst for a large retailer. "
    "Answer the user's question about supplier risk and stockout exposure. "
    "Use ONLY the provided context (supplier snapshot + retrieved evidence). "
    "If the user's assumption conflicts with the snapshot (e.g. they say 'high "
    "risk' but the band is 'medium'), correct them clearly. Cite evidence by its "
    "[number]. If the context lacks the answer, say so rather than guessing. "
    "Be concise, specific, and decision-oriented for a supply-chain manager."
)


def generate_supplier_risk_answer(
    question: str,
    snapshot_result: dict,
    cleaned_evidence_result: dict
) -> dict:
    """LLM-powered, RAG-grounded answer. Falls back to deterministic on any error."""
    if not snapshot_result.get("ok"):
        return {"ok": False, "error": snapshot_result.get("error", "Supplier snapshot failed."), "answer": None}
    if not cleaned_evidence_result.get("ok"):
        return {"ok": False, "error": cleaned_evidence_result.get("error", "Evidence retrieval failed."), "answer": None}

    snapshot = snapshot_result["snapshot"]
    snapshot_json = snapshot.get("chat_context_json", {}) or {}
    evidence = cleaned_evidence_result.get("evidence", []) or []

    context = _build_grounding_context(snapshot, snapshot_json, evidence)

    if USE_LLM:
        try:
            client = _get_llm_client()
            resp = client.chat.completions.create(
                model=LLM_ENDPOINT,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",
                     "content": f"CONTEXT:\n{context}\n\nQUESTION:\n{question}"},
                ],
                max_tokens=LLM_MAX_TOKENS,
                temperature=LLM_TEMPERATURE,
            )
            answer = resp.choices[0].message.content
            return {
                "ok": True,
                "error": None,
                "answer": answer,
                "supplier_id": snapshot.get("supplier_id"),
                "supplier_name": snapshot.get("supplier_name"),
                "risk_band": snapshot.get("risk_band"),
                "recommended_action": snapshot.get("recommended_action"),
                "top_risk_driver": snapshot.get("top_risk_driver"),
                "generation_mode": "llm_rag",
                "llm_endpoint": LLM_ENDPOINT,
                "evidence": evidence,
            }
        except Exception as e:
            # Fall through to deterministic if the LLM endpoint is unavailable.
            print(f"[generate_supplier_risk_answer] LLM call failed, using deterministic fallback: {e}")

    # ---- Deterministic fallback (your original v1 generator, trimmed) ----
    sid = snapshot.get("supplier_id")
    name = snapshot.get("supplier_name")
    band = snapshot.get("risk_band")
    q_lower = str(question).lower()
    opening = (f"{name} is currently classified as {band}, not high risk."
               if "high risk" in q_lower and str(band).lower() != "high"
               else f"{name} is currently classified as {band}.")
    return {
        "ok": True,
        "error": None,
        "answer": f"{opening}\n\n{context}",
        "supplier_id": sid,
        "supplier_name": name,
        "risk_band": band,
        "recommended_action": snapshot.get("recommended_action"),
        "top_risk_driver": snapshot.get("top_risk_driver"),
        "generation_mode": "deterministic_fallback",
        "evidence": evidence,
    }

In [0]:
# ─────────────────────────────────────────────────────────────
# Cell 8 — End-to-end supplier risk agent runner
# Uses tools from Cells 2–7:
# 1. resolve_supplier_from_question
# 2. get_supplier_snapshot
# 3. retrieve_supplier_evidence
# 4. clean_retrieved_evidence
# 5. generate_supplier_risk_answer
# ─────────────────────────────────────────────────────────────

from datetime import datetime
import json
import traceback

def run_supplier_risk_agent(question: str) -> dict:
    """
    End-to-end v1 supplier risk agent.
    This gives us a stable runner before wiring LangGraph nodes.
    """

    run_started_at = datetime.utcnow()

    agent_trace = []

    try:
        # Step 1: Resolve supplier
        resolver_result = resolve_supplier_from_question(question)

        agent_trace.append({
            "step": "resolve_supplier",
            "ok": resolver_result.get("supplier_id") is not None,
            "method": resolver_result.get("resolution_method"),
            "confidence": resolver_result.get("confidence"),
            "message": resolver_result.get("message")
        })

        if resolver_result.get("supplier_id") is None:
            answer = (
                "I could not identify a specific supplier from the question. "
                "Please include a supplier ID like SUP_134 or a supplier name like FlexPack Industries."
            )

            return {
                "ok": False,
                "error": "supplier_unresolved",
                "question": question,
                "supplier_id": None,
                "supplier_name": None,
                "answer": answer,
                "trace": agent_trace,
                "run_started_at": run_started_at.isoformat(),
                "run_finished_at": datetime.utcnow().isoformat()
            }

        supplier_id = resolver_result["supplier_id"]
        supplier_name = resolver_result["supplier_name"]

        # Step 2: Get supplier snapshot
        snapshot_result = get_supplier_snapshot(supplier_id)

        agent_trace.append({
            "step": "get_supplier_snapshot",
            "ok": snapshot_result.get("ok"),
            "error": snapshot_result.get("error")
        })

        if not snapshot_result.get("ok"):
            return {
                "ok": False,
                "error": snapshot_result.get("error"),
                "question": question,
                "supplier_id": supplier_id,
                "supplier_name": supplier_name,
                "answer": f"I found supplier {supplier_id}, but could not load its chat context snapshot.",
                "trace": agent_trace,
                "run_started_at": run_started_at.isoformat(),
                "run_finished_at": datetime.utcnow().isoformat()
            }

        # Step 3: Retrieve evidence
        evidence_result = retrieve_supplier_evidence(
            supplier_id=supplier_id,
            question=question,
            top_k_candidates=TOP_K_CANDIDATES,
            top_k_final=TOP_K_FINAL
        )

        agent_trace.append({
            "step": "retrieve_supplier_evidence",
            "ok": evidence_result.get("ok"),
            "error": evidence_result.get("error"),
            "candidate_count": evidence_result.get("candidate_count"),
            "returned_count": evidence_result.get("returned_count")
        })

        if not evidence_result.get("ok"):
            return {
                "ok": False,
                "error": evidence_result.get("error"),
                "question": question,
                "supplier_id": supplier_id,
                "supplier_name": supplier_name,
                "answer": f"I loaded the supplier snapshot for {supplier_name}, but could not retrieve supporting evidence.",
                "trace": agent_trace,
                "run_started_at": run_started_at.isoformat(),
                "run_finished_at": datetime.utcnow().isoformat()
            }

        # Step 4: Clean evidence
        cleaned_evidence_result = clean_retrieved_evidence(evidence_result)

        agent_trace.append({
            "step": "clean_retrieved_evidence",
            "ok": cleaned_evidence_result.get("ok"),
            "error": cleaned_evidence_result.get("error"),
            "returned_count": cleaned_evidence_result.get("returned_count")
        })

        # Step 5: Generate answer
        answer_result = generate_supplier_risk_answer(
            question=question,
            snapshot_result=snapshot_result,
            cleaned_evidence_result=cleaned_evidence_result
        )

        agent_trace.append({
            "step": "generate_supplier_risk_answer",
            "ok": answer_result.get("ok"),
            "error": answer_result.get("error"),
            "evidence_count": answer_result.get("evidence_count")
        })

        run_finished_at = datetime.utcnow()

        return {
            "ok": answer_result.get("ok"),
            "error": answer_result.get("error"),
            "question": question,
            "supplier_id": supplier_id,
            "supplier_name": supplier_name,
            "answer": answer_result.get("answer"),
            "risk_band": answer_result.get("risk_band"),
            "recommended_action": answer_result.get("recommended_action"),
            "evidence_count": answer_result.get("evidence_count"),
            "resolver": resolver_result,
            "snapshot": snapshot_result.get("snapshot"),
            "evidence": cleaned_evidence_result.get("evidence"),
            "trace": agent_trace,
            "run_started_at": run_started_at.isoformat(),
            "run_finished_at": run_finished_at.isoformat(),
            "duration_seconds": (run_finished_at - run_started_at).total_seconds()
        }

    except Exception as e:
        run_finished_at = datetime.utcnow()

        return {
            "ok": False,
            "error": str(e),
            "question": question,
            "supplier_id": None,
            "supplier_name": None,
            "answer": "The supplier risk agent encountered an unexpected error.",
            "trace": agent_trace,
            "exception_traceback": traceback.format_exc(),
            "run_started_at": run_started_at.isoformat(),
            "run_finished_at": run_finished_at.isoformat(),
            "duration_seconds": (run_finished_at - run_started_at).total_seconds()
        }


# Test end-to-end runner
agent_test_question = """
Why is supplier SUP_134 high risk?
Explain the main external events, affected SKUs, and recommended action.
"""

agent_result = run_supplier_risk_agent(agent_test_question)

print("Agent runner result:")
print("ok:", agent_result["ok"])
print("error:", agent_result["error"])
print("supplier_id:", agent_result["supplier_id"])
print("supplier_name:", agent_result["supplier_name"])
print("duration_seconds:", agent_result.get("duration_seconds"))

print("\nTrace:")
for step in agent_result["trace"]:
    print(step)

print("\n" + "=" * 100)
print(agent_result["answer"])

In [0]:
# ─────────────────────────────────────────────────────────────
# Cell 9 — Manual LangGraph-style graph nodes
# Why:
# Databricks package environment is throwing TypedDict/langgraph conflicts.
# This keeps the same graph architecture without depending on the broken package.
# ─────────────────────────────────────────────────────────────

from datetime import datetime, timezone
import traceback

def utc_now_iso():
    return datetime.now(timezone.utc).isoformat()

def node_resolve_supplier(state: dict) -> dict:
    question = state.get("question", "")
    resolver_result = resolve_supplier_from_question(question)

    trace = state.get("trace", [])
    trace.append({
        "node": "resolve_supplier",
        "ok": resolver_result.get("supplier_id") is not None,
        "method": resolver_result.get("resolution_method"),
        "confidence": resolver_result.get("confidence"),
        "message": resolver_result.get("message")
    })

    state["resolver"] = resolver_result
    state["supplier_id"] = resolver_result.get("supplier_id")
    state["supplier_name"] = resolver_result.get("supplier_name")
    state["trace"] = trace

    if resolver_result.get("supplier_id") is None:
        state["ok"] = False
        state["error"] = "supplier_unresolved"
        state["final_answer"] = (
            "I could not identify a specific supplier from the question. "
            "Please include a supplier ID like SUP_134 or a supplier name like FlexPack Industries."
        )

    return state


def node_get_snapshot(state: dict) -> dict:
    if state.get("error"):
        return state

    supplier_id = state.get("supplier_id")
    snapshot_result = get_supplier_snapshot(supplier_id)

    trace = state.get("trace", [])
    trace.append({
        "node": "get_supplier_snapshot",
        "ok": snapshot_result.get("ok"),
        "error": snapshot_result.get("error")
    })

    state["snapshot_result"] = snapshot_result
    state["trace"] = trace

    if not snapshot_result.get("ok"):
        state["ok"] = False
        state["error"] = snapshot_result.get("error")
        state["final_answer"] = (
            f"I found supplier {supplier_id}, but could not load its chat context snapshot."
        )

    return state


def node_retrieve_evidence(state: dict) -> dict:
    if state.get("error"):
        return state

    evidence_result = retrieve_supplier_evidence(
        supplier_id=state.get("supplier_id"),
        question=state.get("question"),
        top_k_candidates=TOP_K_CANDIDATES,
        top_k_final=TOP_K_FINAL
    )

    trace = state.get("trace", [])
    trace.append({
        "node": "retrieve_supplier_evidence",
        "ok": evidence_result.get("ok"),
        "error": evidence_result.get("error"),
        "candidate_count": evidence_result.get("candidate_count"),
        "returned_count": evidence_result.get("returned_count")
    })

    state["evidence_result"] = evidence_result
    state["trace"] = trace

    if not evidence_result.get("ok"):
        state["ok"] = False
        state["error"] = evidence_result.get("error")
        state["final_answer"] = (
            f"I loaded the supplier snapshot for {state.get('supplier_name')}, "
            "but could not retrieve supporting evidence."
        )

    return state


def node_clean_evidence(state: dict) -> dict:
    if state.get("error"):
        return state

    cleaned_evidence_result = clean_retrieved_evidence(
        state.get("evidence_result")
    )

    trace = state.get("trace", [])
    trace.append({
        "node": "clean_retrieved_evidence",
        "ok": cleaned_evidence_result.get("ok"),
        "error": cleaned_evidence_result.get("error"),
        "returned_count": cleaned_evidence_result.get("returned_count")
    })

    state["cleaned_evidence_result"] = cleaned_evidence_result
    state["trace"] = trace

    if not cleaned_evidence_result.get("ok"):
        state["ok"] = False
        state["error"] = cleaned_evidence_result.get("error")
        state["final_answer"] = "I retrieved evidence, but could not clean it for answer generation."

    return state


def node_generate_answer(state: dict) -> dict:
    if state.get("error"):
        return state

    answer_result = generate_supplier_risk_answer(
        question=state.get("question"),
        snapshot_result=state.get("snapshot_result"),
        cleaned_evidence_result=state.get("cleaned_evidence_result")
    )

    trace = state.get("trace", [])
    trace.append({
        "node": "generate_supplier_risk_answer",
        "ok": answer_result.get("ok"),
        "error": answer_result.get("error"),
        "evidence_count": answer_result.get("evidence_count")
    })

    state["answer_result"] = answer_result
    state["trace"] = trace

    if answer_result.get("ok"):
        state["ok"] = True
        state["error"] = None
        state["final_answer"] = answer_result.get("answer")
        state["risk_band"] = answer_result.get("risk_band")
        state["recommended_action"] = answer_result.get("recommended_action")
        state["evidence_count"] = answer_result.get("evidence_count")
    else:
        state["ok"] = False
        state["error"] = answer_result.get("error")
        state["final_answer"] = "I could not generate a supplier risk answer."

    return state


def run_supplier_risk_graph(question: str) -> dict:
    """
    Manual LangGraph-style graph runner.
    Same node architecture, no external LangGraph dependency.
    """

    run_started_at = datetime.now(timezone.utc)

    state = {
        "question": question,
        "ok": None,
        "error": None,
        "trace": [],
        "run_started_at": run_started_at.isoformat(),
        "run_finished_at": None,
        "duration_seconds": None
    }

    try:
        for node_fn in [
            node_resolve_supplier,
            node_get_snapshot,
            node_retrieve_evidence,
            node_clean_evidence,
            node_generate_answer
        ]:
            state = node_fn(state)

            # Stop early if a node set an error
            if state.get("error"):
                break

    except Exception as e:
        state["ok"] = False
        state["error"] = str(e)
        state["final_answer"] = "The supplier risk graph encountered an unexpected error."
        state["exception_traceback"] = traceback.format_exc()

    run_finished_at = datetime.now(timezone.utc)

    state["run_finished_at"] = run_finished_at.isoformat()
    state["duration_seconds"] = (run_finished_at - run_started_at).total_seconds()

    return state


# Test manual graph
graph_test_question = """
Why is supplier SUP_134 high risk?
Explain the main external events, affected SKUs, and recommended action.
"""

graph_result = run_supplier_risk_graph(graph_test_question)

print("Manual graph result:")
print("ok:", graph_result.get("ok"))
print("error:", graph_result.get("error"))
print("supplier_id:", graph_result.get("supplier_id"))
print("supplier_name:", graph_result.get("supplier_name"))
print("duration_seconds:", graph_result.get("duration_seconds"))

print("\nGraph trace:")
for step in graph_result.get("trace", []):
    print(step)

print("\n" + "=" * 100)
print(graph_result.get("final_answer"))

In [0]:
# ─────────────────────────────────────────────────────────────
# Cell 10 — Save supplier risk agent run log
# ─────────────────────────────────────────────────────────────

import json
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    IntegerType,
    BooleanType,
    TimestampType
)

AGENT_RUN_LOG_TABLE = "supplysage_gold.gold_supplier_risk_agent_run_logs"

assert "graph_result" in globals(), "graph_result not found. Rerun Cell 9."

def safe_json_dumps(obj):
    return json.dumps(obj, default=str, ensure_ascii=False)

run_log_row = [{
    "agent_run_id": f"agent_run_{graph_result.get('supplier_id')}_{datetime.now(timezone.utc).strftime('%Y%m%d%H%M%S')}",
    "question": graph_result.get("question"),
    "ok": bool(graph_result.get("ok")),
    "error": graph_result.get("error"),
    "supplier_id": graph_result.get("supplier_id"),
    "supplier_name": graph_result.get("supplier_name"),
    "risk_band": graph_result.get("risk_band"),
    "recommended_action": graph_result.get("recommended_action"),
    "evidence_count": int(graph_result.get("evidence_count") or 0),
    "duration_seconds": float(graph_result.get("duration_seconds") or 0.0),
    "trace_json": safe_json_dumps(graph_result.get("trace")),
    "resolver_json": safe_json_dumps(graph_result.get("resolver")),
    "snapshot_json": safe_json_dumps(graph_result.get("snapshot_result")),
    "evidence_json": safe_json_dumps(graph_result.get("cleaned_evidence_result")),
    "final_answer": graph_result.get("final_answer"),
    "run_started_at": graph_result.get("run_started_at"),
    "run_finished_at": graph_result.get("run_finished_at"),
    "logged_at": datetime.now(timezone.utc)
}]

run_log_schema = StructType([
    StructField("agent_run_id", StringType(), False),
    StructField("question", StringType(), True),
    StructField("ok", BooleanType(), True),
    StructField("error", StringType(), True),
    StructField("supplier_id", StringType(), True),
    StructField("supplier_name", StringType(), True),
    StructField("risk_band", StringType(), True),
    StructField("recommended_action", StringType(), True),
    StructField("evidence_count", IntegerType(), True),
    StructField("duration_seconds", DoubleType(), True),
    StructField("trace_json", StringType(), True),
    StructField("resolver_json", StringType(), True),
    StructField("snapshot_json", StringType(), True),
    StructField("evidence_json", StringType(), True),
    StructField("final_answer", StringType(), True),
    StructField("run_started_at", StringType(), True),
    StructField("run_finished_at", StringType(), True),
    StructField("logged_at", TimestampType(), True),
])

run_log_df = spark.createDataFrame(run_log_row, schema=run_log_schema)

(
    run_log_df
    .withColumn("gold_created_at", F.current_timestamp())
    .withColumn("gold_source_notebook", F.lit("27_langgraph_supplier_risk_agent"))
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(AGENT_RUN_LOG_TABLE)
)

print(f"Saved agent run log to: {AGENT_RUN_LOG_TABLE}")

display(
    spark.table(AGENT_RUN_LOG_TABLE)
    .select(
        "agent_run_id",
        "supplier_id",
        "supplier_name",
        "ok",
        "risk_band",
        "recommended_action",
        "evidence_count",
        "duration_seconds",
        "logged_at"
    )
    .orderBy(F.desc("logged_at"))
    .limit(10)
)